A notebook to test every function in robots classes.

In [ ]:
import time
import logging
import numpy as np
from robots.dual_dobot.config_dobot import DobotDualArmConfig
from robots.dual_dobot.dobot_dual_arm import DobotDualArm

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
log = logging.getLogger(__name__)

# 创建配置对象
config = DobotDualArmConfig(
    name="dobot_dual_arm",
    robot_port=4242,
    gripper_port=4243,
    use_gripper=True,
    gripper_max_open=0.085,
    gripper_force=10.0,
    gripper_speed=0.1,
    gripper_reverse=False,
    close_threshold=0.5,
    control_mode="oculus",
    debug=False,
    num_joints_per_arm=6,
    cameras={},
    max_joint_velocity=2.0,
    max_ee_velocity=0.5,
    max_joint_delta=0.3
)

robot = None

# 1. 初始化机器人
log.info("\n--- 1. 初始化机器人 ---")
robot = DobotDualArm(config)
log.info("机器人对象创建成功")

# 2. 连接机器人
log.info("\n--- 2. 连接机器人 ---")
robot.connect()
log.info("机器人连接成功")

In [ ]:
# 3. 获取初始观测
log.info("\n--- 3. 获取初始观测 ---")
obs = robot.get_observation()
log.info("成功获取观测数据")
log.info(f"左臂末端位姿: {[obs[f'left_ee_pose.{axis}'] for axis in ['x', 'y', 'z', 'rx', 'ry', 'rz']]}")
log.info(f"右臂末端位姿: {[obs[f'right_ee_pose.{axis}'] for axis in ['x', 'y', 'z', 'rx', 'ry', 'rz']]}")
log.info(f"左臂关节位置: {[obs[f'left_joint_{i+1}.pos'] for i in range(config.num_joints_per_arm)]}")
log.info(f"右臂关节位置: {[obs[f'right_joint_{i+1}.pos'] for i in range(config.num_joints_per_arm)]}")
if config.use_gripper:
    log.info(f"左夹爪状态: {obs['left_gripper_state_norm']}")
    log.info(f"右夹爪状态: {obs['right_gripper_state_norm']}")
time.sleep(1)
        
# 4. 测试重置功能
log.info("\n--- 4. 测试重置功能 ---")
robot.reset()
log.info("机器人重置成功")
time.sleep(2)

init_left_joint_positions = np.asarray([obs[f'left_joint_{i+1}.pos'] for i in range(config.num_joints_per_arm)])
print("初始左臂关节位置:", init_left_joint_positions)   
init_right_joint_positions = np.asarray([obs[f'right_joint_{i+1}.pos'] for i in range(config.num_joints_per_arm)])
print("初始右臂关节位置:", init_right_joint_positions) 


In [ ]:
robot.left

In [ ]:
start_time = time.perf_counter()
obs = robot.get_observation()
end_time = time.perf_counter()
print(f"get_observation time: {end_time - start_time}")
left_joint_positions = np.asarray([obs[f'left_joint_{i+1}.pos'] for i in range(config.num_joints_per_arm)])
right_joint_positions = np.asarray([obs[f'right_joint_{i+1}.pos'] for i in range(config.num_joints_per_arm)])
for i in range (10):
    # obs = robot.get_observation()
    # left_joint_positions = np.asarray([obs[f'left_joint_{i+1}.pos'] for i in range(config.num_joints_per_arm)])
    # right_joint_positions = np.asarray([obs[f'right_joint_{i+1}.pos'] for i in range(config.num_joints_per_arm)])
    left_joint_positions[0] += 0.02
    right_joint_positions[0] += 0.02
    action = {
        "left_joint_1.pos": left_joint_positions[0]+0.05,
        "left_joint_2.pos": left_joint_positions[1],
        "left_joint_3.pos": left_joint_positions[2],
        "left_joint_4.pos": left_joint_positions[3],
        "left_joint_5.pos": left_joint_positions[4],
        "left_joint_6.pos": left_joint_positions[5],
        "right_joint_1.pos": right_joint_positions[0]+0.05,
        "right_joint_2.pos": right_joint_positions[1],
        "right_joint_3.pos": right_joint_positions[2],
        "right_joint_4.pos": right_joint_positions[3],
        "right_joint_5.pos": right_joint_positions[4],
        "right_joint_6.pos": right_joint_positions[5]
    }
    robot.send_action(action)
    time.sleep(0.05)

In [ ]:
#!/usr/bin/env python
"""
Test script for IK flow:
1. Get current joint positions from robot
2. Compute current EE pose via placo FK
3. Set a target EE pose (small offset from current)
4. Solve IK to get target joint positions
5. Send action to robot
6. Repeat 100 steps
"""

import time
import numpy as np
from scipy.spatial.transform import Rotation as R
import placo

# Import robot client
from lerobot_robot.dobot_interface_client import DobotDualArmClient

# URDF path
URDF_PATH = "/home/geist/lerobot/dual_arm_data_collection/lerobot_dual_arm_teleop/assets/dual_dobot/dual_nova5_robot.urdf"

# Joint indices in placo model
LEFT_ARM_Q_IDX = slice(7, 13)   # q[7:13]
RIGHT_ARM_Q_IDX = slice(25, 31) # q[25:31]


def create_ik_solver(urdf_path: str):
    """Create and configure placo IK solver."""
    robot = placo.RobotWrapper(urdf_path)
    solver = placo.KinematicsSolver(robot)
    solver.dt = 0.01
    solver.mask_fbase(True)
    
    # Add regularization
    solver.add_kinetic_energy_regularization_task(1e-6)
    joint_reg = solver.add_regularization_task(0.1)
    joint_reg.configure("joint_regularization", "soft", 0.1)
    
    # Enable joint limits
    solver.enable_joint_limits(True)
    
    return robot, solver


def create_frame_task(solver, robot, ee_link: str, initial_pose: np.ndarray):
    """Create a frame task for end-effector."""
    task = solver.add_frame_task(ee_link, initial_pose)
    task.configure(f"{ee_link}_frame", "soft", 1.0)
    return task


def get_current_ee_pose(robot, ee_link: str) -> tuple:
    """Get current end-effector pose from placo FK."""
    T_ee = robot.get_T_world_frame(ee_link)
    pos = T_ee[:3, 3].copy()
    rot = R.from_matrix(T_ee[:3, :3]).as_euler("xyz")
    return pos, rot, T_ee


def compute_target_pose(current_pos: np.ndarray, current_rot: np.ndarray, 
                        delta_pos: np.ndarray, delta_rot: np.ndarray) -> tuple:
    """Compute target pose by adding delta to current pose."""
    target_pos = current_pos + delta_pos
    
    R_current = R.from_euler("xyz", current_rot)
    R_delta = R.from_euler("xyz", delta_rot)
    R_target = R_delta * R_current
    target_rot = R_target.as_euler("xyz")
    
    # Build transform matrix
    T_target = np.eye(4)
    T_target[:3, 3] = target_pos
    T_target[:3, :3] = R_target.as_matrix()
    
    return target_pos, target_rot, T_target


print("=" * 60)
print("IK Flow Test")
print("=" * 60)

# Connect to robot
print("\n[1] Connecting to robot...")
robot_client = DobotDualArmClient(ip='127.0.0.1', port=4242)

# Create IK solver
print("[2] Creating IK solver...")
placo_robot, solver = create_ik_solver(URDF_PATH)

robot_client.robot_go_home()

In [ ]:
# ServoJ
# Create frame tasks
print("[3] Creating frame tasks...")
initial_pose = np.eye(4)
left_task = create_frame_task(solver, placo_robot, "left_Link6", initial_pose)
right_task = create_frame_task(solver, placo_robot, "right_Link6", initial_pose)

# Get initial joint positions
print("[4] Getting initial joint positions...")
left_joints = robot_client.left_robot_get_joint_positions()
right_joints = robot_client.right_robot_get_joint_positions()
print(f"    Left joints:  {np.rad2deg(left_joints).round(2)} deg")
print(f"    Right joints: {np.rad2deg(right_joints).round(2)} deg")

# Set initial joints in placo
placo_robot.state.q[LEFT_ARM_Q_IDX] = left_joints
placo_robot.state.q[RIGHT_ARM_Q_IDX] = right_joints
placo_robot.update_kinematics()

# Get initial EE poses
left_pos, left_rot, _ = get_current_ee_pose(placo_robot, "left_Link6")
right_pos, right_rot, _ = get_current_ee_pose(placo_robot, "right_Link6")
print(f"\n    Left EE:  pos={left_pos.round(4)}, rot={np.rad2deg(left_rot).round(2)} deg")
print(f"    Right EE: pos={right_pos.round(4)}, rot={np.rad2deg(right_rot).round(2)} deg")

# Define small delta for testing
delta_pos = np.array([0.01, 0.00, 0.00])  # 1cm forward
delta_rot = np.array([0.1, 0.02, 0.00])  # 1 deg yaw

print(f"\n[5] Delta: pos={delta_pos}m, rot={np.rad2deg(delta_rot)} deg")

# Main loop
print("\n[6] Starting test loop (100 steps)...")
print("-" * 60)

for step in range(100):
    t_start = time.perf_counter()
    
    # --- Step 1: Get current joint positions from robot ---
    t1 = time.perf_counter()
    left_joints = robot_client.left_robot_get_joint_positions()
    right_joints = robot_client.right_robot_get_joint_positions()
    # print(f"Step {step:3d}: Left joints:  {np.rad2deg(left_joints).round(2)} deg")
    t_get_joints = (time.perf_counter() - t1) * 1000
    
    # --- Step 2: Update placo model and get current EE pose ---
    t2 = time.perf_counter()
    placo_robot.state.q[LEFT_ARM_Q_IDX] = left_joints
    placo_robot.state.q[RIGHT_ARM_Q_IDX] = right_joints
    placo_robot.update_kinematics()
    
    left_pos, left_rot, _ = get_current_ee_pose(placo_robot, "left_Link6")
    right_pos, right_rot, _ = get_current_ee_pose(placo_robot, "right_Link6")
    t_fk = (time.perf_counter() - t2) * 1000
    
    # --- Step 3: Compute target pose ---
    t3 = time.perf_counter()
    _, _, T_left_target = compute_target_pose(left_pos, left_rot, delta_pos, delta_rot)
    _, _, T_right_target = compute_target_pose(right_pos, right_rot, delta_pos, delta_rot)
    t_target = (time.perf_counter() - t3) * 1000
    
    # --- Step 4: Solve IK ---
    t4 = time.perf_counter()
    left_task.T_world_frame = T_left_target
    right_task.T_world_frame = T_right_target
    solver.solve(True)
    
    target_left_q = placo_robot.state.q[LEFT_ARM_Q_IDX].copy()
    target_right_q = placo_robot.state.q[RIGHT_ARM_Q_IDX].copy()
    t_ik = (time.perf_counter() - t4) * 1000
    
    # --- Step 5: Send action to robot ---
    t5 = time.perf_counter()
    robot_client.servo_j('left', np.asarray(target_left_q), t=0.05, lookahead_time=0.05, gain=300)
    robot_client.servo_j('right', np.asarray(target_right_q), t=0.05, lookahead_time=0.05, gain=300)
    t_send = (time.perf_counter() - t5) * 1000
    
    # --- Timing summary ---
    t_total = (time.perf_counter() - t_start) * 1000
    
    if step % 10 == 0:
        print(f"Step {step:3d}: "
                f"get_joints={t_get_joints:5.1f}ms, "
                f"FK={t_fk:5.1f}ms, "
                f"target={t_target:5.1f}ms, "
                f"IK={t_ik:5.1f}ms, "
                f"send={t_send:5.1f}ms, "
                f"total={t_total:5.1f}ms")
    
    # Small delay to not overwhelm the robot
    if t_total < 50:
        time.sleep((50 - t_total)/1000)
        # print("sleep")

print("-" * 60)
print("\n[7] Test completed!")

# Get final positions
left_joints = robot_client.left_robot_get_joint_positions()
right_joints = robot_client.right_robot_get_joint_positions()
placo_robot.state.q[LEFT_ARM_Q_IDX] = left_joints
placo_robot.state.q[RIGHT_ARM_Q_IDX] = right_joints
placo_robot.update_kinematics()

left_pos, left_rot, _ = get_current_ee_pose(placo_robot, "left_Link6")
right_pos, right_rot, _ = get_current_ee_pose(placo_robot, "right_Link6")

print(f"\nFinal Left EE:  pos={left_pos.round(4)}, rot={np.rad2deg(left_rot).round(2)} deg")
print(f"Final Right EE: pos={right_pos.round(4)}, rot={np.rad2deg(right_rot).round(2)} deg")


In [ ]:
# ServoP
# Create frame tasks
print("[3] Creating frame tasks...")
initial_pose = np.eye(4)
left_task = create_frame_task(solver, placo_robot, "left_Link6", initial_pose)
right_task = create_frame_task(solver, placo_robot, "right_Link6", initial_pose)

# Get initial joint positions
print("[4] Getting initial joint positions...")
left_joints = robot_client.left_robot_get_joint_positions()
right_joints = robot_client.right_robot_get_joint_positions()
print(f"    Left joints:  {np.rad2deg(left_joints).round(2)} deg")
print(f"    Right joints: {np.rad2deg(right_joints).round(2)} deg")

# Set initial joints in placo
placo_robot.state.q[LEFT_ARM_Q_IDX] = left_joints
placo_robot.state.q[RIGHT_ARM_Q_IDX] = right_joints
placo_robot.update_kinematics()

# Get initial EE poses
left_pos, left_rot, _ = get_current_ee_pose(placo_robot, "left_Link6")
right_pos, right_rot, _ = get_current_ee_pose(placo_robot, "right_Link6")
print(f"\n    Left EE:  pos={left_pos.round(4)}, rot={np.rad2deg(left_rot).round(2)} deg")
print(f"    Right EE: pos={right_pos.round(4)}, rot={np.rad2deg(right_rot).round(2)} deg")

# Define small delta for testing (move forward 1cm, rotate 1 deg)
delta_pos = np.array([0.01, 0.0, 0.0])  # 1cm forward
delta_rot = np.array([0.0, 0.0, np.deg2rad(1.0)])  # 1 deg yaw

print(f"\n[5] Delta: pos={delta_pos}m, rot={np.rad2deg(delta_rot)} deg")

# Main loop
print("\n[6] Starting test loop (100 steps)...")
print("-" * 60)

for step in range(100):
    t_start = time.perf_counter()
    
    # --- Step 1: Get current joint positions from robot ---
    t1 = time.perf_counter()
    left_joints = robot_client.left_robot_get_joint_positions()
    right_joints = robot_client.right_robot_get_joint_positions()
    t_get_joints = (time.perf_counter() - t1) * 1000
    
    # --- Step 2: Update placo model and get current EE pose ---
    t2 = time.perf_counter()
    placo_robot.state.q[LEFT_ARM_Q_IDX] = left_joints
    placo_robot.state.q[RIGHT_ARM_Q_IDX] = right_joints
    placo_robot.update_kinematics()
    
    left_pos, left_rot, _ = get_current_ee_pose(placo_robot, "left_Link6")
    right_pos, right_rot, _ = get_current_ee_pose(placo_robot, "right_Link6")
    t_fk = (time.perf_counter() - t2) * 1000
    
    # --- Step 3: Compute target pose ---
    t3 = time.perf_counter()
    _, _, T_left_target = compute_target_pose(left_pos, left_rot, delta_pos, delta_rot)
    _, _, T_right_target = compute_target_pose(right_pos, right_rot, delta_pos, delta_rot)
    t_target = (time.perf_counter() - t3) * 1000
    
    # --- Step 4: Solve IK ---
    t4 = time.perf_counter()
    left_task.T_world_frame = T_left_target
    right_task.T_world_frame = T_right_target
    solver.solve(True)
    
    target_left_q = placo_robot.state.q[LEFT_ARM_Q_IDX].copy()
    target_right_q = placo_robot.state.q[RIGHT_ARM_Q_IDX].copy()
    t_ik = (time.perf_counter() - t4) * 1000
    
    # --- Step 5: Send action to robot ---
    t5 = time.perf_counter()
    robot_client.servo_j('left', np.asarray(target_left_q), t=0.05, lookahead_time=0.05, gain=300)
    robot_client.servo_j('right', np.asarray(target_right_q), t=0.05, lookahead_time=0.05, gain=300)
    t_send = (time.perf_counter() - t5) * 1000
    
    # --- Timing summary ---
    t_total = (time.perf_counter() - t_start) * 1000
    
    if step % 10 == 0:
        print(f"Step {step:3d}: "
                f"get_joints={t_get_joints:5.1f}ms, "
                f"FK={t_fk:5.1f}ms, "
                f"target={t_target:5.1f}ms, "
                f"IK={t_ik:5.1f}ms, "
                f"send={t_send:5.1f}ms, "
                f"total={t_total:5.1f}ms")
    
    # Small delay to not overwhelm the robot
    time.sleep(0.01)

print("-" * 60)
print("\n[7] Test completed!")

# Get final positions
left_joints = robot_client.left_robot_get_joint_positions()
right_joints = robot_client.right_robot_get_joint_positions()
placo_robot.state.q[LEFT_ARM_Q_IDX] = left_joints
placo_robot.state.q[RIGHT_ARM_Q_IDX] = right_joints
placo_robot.update_kinematics()

left_pos, left_rot, _ = get_current_ee_pose(placo_robot, "left_Link6")
right_pos, right_rot, _ = get_current_ee_pose(placo_robot, "right_Link6")

print(f"\nFinal Left EE:  pos={left_pos.round(4)}, rot={np.rad2deg(left_rot).round(2)} deg")
print(f"Final Right EE: pos={right_pos.round(4)}, rot={np.rad2deg(right_rot).round(2)} deg")
